# Reasoning Agent: Natural Language → Logic → Classical Reasoning → True/False

Pipeline: a router sends each (context, statement) pair to the cheapest sound route:
1. **causal**: causal story → structural causal model → counterfactual by abduction/action/prediction
2. **horn**: deterministic grammar → Horn clauses with negation → tabled backward chaining (closed world; agrees with forward chaining / well-founded semantics on every public question)
3. **hybrid**: grammar + LLM translation of unparsed sentences → Horn clauses → backward chaining
4. **fol**: LLM translation to first-order logic (k samples) → our resolution prover → majority vote of prover verdicts

The LLM (optional, local) is used only as a semantic parser. Every True/False decision is made by the classical engines below.

In [1]:
# ---- configuration ----
import os, json, time, csv
DATA_DIR = '../data'
TEST_FILE = None          # CSV/JSONL with columns id, context, statement -> writes submission.csv
USE_LLM = False           # True: load the local LLM for sentences the grammar cannot parse
LLM_MODEL = 'Qwen/Qwen2.5-14B-Instruct'   # bf16 base of the LoRA adapter
LLM_TP = 2                # tensor-parallel GPUs (2x24 GB A30; 1 is enough on a 48 GB GPU)
LLM_ADAPTER = '/scratch/r25cs0001/ai-assign/models/lora14b-r1'  # LoRA trained on solver-verified FOLIO-train pairs
K_SAMPLES = 3             # translations per FOL problem; majority vote over prover verdicts
CACHE_FILE = '../cache/translations.json'
os.environ.setdefault('HF_HOME', '/scratch/r25cs0001/ai-assign/hf')

'/scratch/r25cs0001/ai-assign/hf'

## Internal representation
`reasoner/logic.py`

In [2]:
"""Internal representation shared by the Horn-clause parsers and engines.

A literal is a plain tuple ``(pred, arg1, ..., argN)``. Negation is encoded in
the predicate name with a leading ``~``; tuples hash fast, so the fact base is
just a set of literals. Variables are strings starting with ``?``.

  young(bob)          -> ("young", "bob")
  not chase(cat, cow) -> ("~chase", "cat", "cow")
"""
from dataclasses import dataclass, field


def is_var(term):
    return term[0] == "?"


def negate(lit):
    p = lit[0]
    return (p[1:],) + lit[1:] if p[0] == "~" else ("~" + p,) + lit[1:]


def is_neg(lit):
    return lit[0][0] == "~"


def substitute(lit, theta):
    return (lit[0],) + tuple(theta.get(t, t) for t in lit[1:])


def lit_str(lit):
    p = lit[0]
    s = f"{p.lstrip('~')}({', '.join(lit[1:])})"
    return "¬" + s if p[0] == "~" else s


@dataclass(frozen=True)
class Rule:
    body: tuple  # conjunction of literals
    head: tuple  # single literal
    text: str = field(default="", compare=False)

    def __str__(self):
        return " ∧ ".join(map(lit_str, self.body)) + " → " + lit_str(self.head)


@dataclass
class Theory:
    facts: set = field(default_factory=set)
    rules: list = field(default_factory=list)
    unparsed: list = field(default_factory=list)  # sentences no grammar rule matched


## Parser A: RuleTaker-style grammar → Horn clauses
`reasoner/rt_parser.py`

In [3]:
"""Deterministic grammar for RuleTaker-style English -> Horn clauses with negation.

No dataset vocabulary is hard-coded: adjectives, verbs and entity names are
taken from the sentence shape, so unseen names/predicates parse the same way.
"""
import re


VAR = "?x"
VAR_WORDS = {"someone", "something", "it", "they", "he", "she", "them",
             "somebody", "anyone", "anything", "everyone", "everything", "people", "things"}

# Entity shapes: variable word | "the <noun>" (up to 3 words) | Capitalised name.
ENT = r"(?:(?i:someone|something|somebody|anyone|anything|everyone|everything|it|they|he|she|them)" \
      r"|(?i:the)(?: (?!(?:does|do|did|not|is|are|and|or|then|if)\b)[a-z][\w-]*){1,3}|[A-Z][\w-]*)"

RE_COPULA = re.compile(rf"^(?P<s>{ENT}) (?i:is|are) (?P<neg>(?i:not) )?(?P<p>[\w -]+)$")
RE_VERB = re.compile(rf"^(?P<s>{ENT}) (?P<neg>(?i:does not|do not|doesn't|don't) )?"
                     rf"(?P<v>[a-z][\w-]*) (?P<o>{ENT})$")
RE_IF = re.compile(r"^(?i:if) (?P<c>.+?),? (?i:then) (?P<h>.+)$")
RE_ALL = re.compile(r"^(?:(?i:all) )?(?P<adjs>[a-z][\w ,-]*?) (?:things|people|persons|ones|animals) "
                    r"(?:are|is) (?P<rest>.+)$", re.I)
RE_SPLIT_AND = re.compile(r",? and |, ")
RE_SENT = re.compile(r"(?<=[.!?])\s+")


def entity(s):
    low = s.lower()
    if low in VAR_WORDS:
        return VAR
    if low.startswith("the "):
        low = low[4:]
    return low.replace(" ", "_")


def lemma(v):
    """Third-person-singular verb -> base form (chases -> chase, watches -> watch)."""
    if v.endswith(("ches", "shes", "sses", "xes", "zes")):
        return v[:-2]
    if v.endswith("ies") and len(v) > 4:
        return v[:-3] + "y"
    if v.endswith("s") and not v.endswith("ss"):
        return v[:-1]
    return v


def adj_lit(adj, subj, neg):
    p = adj.strip().lower().replace(" ", "_")
    if p.startswith(("a_", "an_")):  # "is a bald eagle" -> bald_eagle
        p = p.split("_", 1)[1]
    return ("~" + p if neg else p, subj)


def parse_atomic(text, prev=None):
    """Parse one clause into a literal. ``prev`` = (subject, kind) of the previous
    conjunct so elliptical chunks like 'red' or 'not furry' reuse it."""
    text = text.strip()
    m = RE_VERB.match(text)
    if m and m.group("v").lower() not in ("is", "are", "not"):
        s, o = entity(m.group("s")), entity(m.group("o"))
        return ("~" if m.group("neg") else "") + lemma(m.group("v").lower()), s, o
    m = RE_COPULA.match(text)
    if m:
        return adj_lit(m.group("p"), entity(m.group("s")), bool(m.group("neg")))
    if prev is not None:
        subj = prev
        m = re.match(r"^(?P<neg>not )?(?P<p>[a-z][\w-]*)$", text, re.I)
        if m:
            return adj_lit(m.group("p"), subj, bool(m.group("neg")))
        m = re.match(rf"^(?P<neg>(?i:does not|do not) )?(?P<v>[a-z][\w-]*) (?P<o>{ENT})$", text)
        if m:
            return ("~" if m.group("neg") else "") + lemma(m.group("v").lower()), subj, entity(m.group("o"))
    return None


def parse_conj(text):
    """'X is a and b and Y chases Z' -> list of literals (None if any chunk fails)."""
    lits, prev = [], None
    for chunk in RE_SPLIT_AND.split(text.strip()):
        lit = parse_atomic(chunk, prev)
        if lit is None:
            return None
        lits.append(lit)
        prev = lit[1]
    return lits


def parse_sentence(sent):
    """Return ('fact', [lits]) | ('rule', [Rule]) | None."""
    s = sent.strip().rstrip(".!").strip()
    if not s:
        return "fact", []
    m = RE_IF.match(s)
    if m:
        body, heads = parse_conj(m.group("c")), parse_conj(m.group("h"))
        if body and heads:
            return "rule", [Rule(tuple(body), h, sent) for h in heads]
        return None
    m = RE_ALL.match(s)
    if m:
        adjs = [a for a in RE_SPLIT_AND.split(m.group("adjs")) if a.strip()]
        heads = parse_conj("it is " + m.group("rest"))
        if adjs and heads and all(re.fullmatch(r"[a-z][\w-]*", a.strip(), re.I) for a in adjs):
            body = tuple(adj_lit(a, VAR, False) for a in adjs)
            return "rule", [Rule(body, h, sent) for h in heads]
    lits = parse_conj(s)
    if lits is None:
        return None
    if any(VAR in l[1:] for l in lits):  # "Something is red" is not a ground fact
        return None
    return "fact", lits


def parse_context(context):
    th = Theory()
    for sent in RE_SENT.split(context.strip()):
        r = parse_sentence(sent)
        if r is None:
            th.unparsed.append(sent)
        elif r[0] == "fact":
            th.facts.update(r[1])
        else:
            th.rules.extend(r[1])
    return th


def parse_query(statement):
    r = parse_sentence(statement)
    if r is None or r[0] != "fact" or len(r[1]) != 1:
        return None
    return r[1][0]


## Engine: forward chaining (OWA / CWA via alternating fixpoint)
`reasoner/forward.py`

In [4]:
"""Forward chaining over Horn rules with negated literals.

mode="owa": a negated literal (~p) is just another fact; it must be stated or derived.
mode="cwa": a negated body literal ~p(t) holds iff p(t) is not derivable (negation as
            failure). Computed with the alternating fixpoint (well-founded semantics):
            Gamma(S) = least model where ~p holds iff p not in S. Iterating
            T <- Gamma(Gamma(T)) from T = {} converges to the true atoms; for programs
            that are stratified at the ground level this is exactly the perfect model,
            without needing a predicate-level stratification (which is too coarse).

Every derived fact records (rule, premises) so proofs can be printed.
"""
from collections import defaultdict



class FactBase:
    def __init__(self, facts):
        self.facts = set()
        self.by_pred = defaultdict(set)
        self.consts = set()
        self.why = {}
        for f in facts:
            self.add(f, None)

    def add(self, f, why):
        if f in self.facts:
            return False
        self.facts.add(f)
        self.by_pred[f[0]].add(f)
        self.consts.update(f[1:])
        self.why[f] = why
        return True


def _matches(body, theta, fb, mode, ref=None):
    """Yield substitutions satisfying ``body``. Positive / OWA literals are joined
    against the fact base first; CWA-negated literals are checked once ground."""
    if not body:
        yield theta
        return
    pos = [l for l in body if mode == "owa" or not is_neg(l)]
    if pos:
        # most-bound literal first: fewer candidate facts
        lit = max(pos, key=lambda l: sum(not is_var(theta.get(t, t)) for t in l[1:]))
        rest = [l for l in body if l is not lit]
        pat = substitute(lit, theta)
        for f in fb.by_pred.get(pat[0], ()):
            th = theta
            for a, b in zip(pat[1:], f[1:]):
                if is_var(a):
                    if th.get(a, b) != b:
                        break
                    if a not in th:
                        th = {**th, a: b}
                elif a != b:
                    break
            else:
                yield from _matches(rest, th, fb, mode, ref)
        return
    # only NAF literals left: bind remaining vars over the domain, then test absence
    lit = body[0]
    free = [t for t in lit[1:] if is_var(theta.get(t, t))]
    if free:
        for c in list(fb.consts):
            yield from _matches(body, {**theta, free[0]: c}, fb, mode, ref)
        return
    if negate(substitute(lit, theta)) not in ref:
        yield from _matches(body[1:], theta, fb, mode, ref)


def _lfp(theory, mode, ref=None, goal=None):
    fb = FactBase(theory.facts)
    rules = [(r, list(r.body)) for r in theory.rules]
    changed = True
    while changed:
        changed = False
        for r, body in rules:
            for theta in list(_matches(body, {}, fb, mode, ref)):
                h = substitute(r.head, theta)
                if any(is_var(t) for t in h[1:]):
                    continue
                if fb.add(h, (r, tuple(substitute(l, theta) for l in r.body))):
                    changed = True
                    if goal is not None and h == goal:
                        return fb
    return fb


def forward_chain(theory, mode="owa", goal=None):
    if mode == "owa":
        return _lfp(theory, mode, goal=goal)
    if not any(is_neg(l) for r in theory.rules for l in r.body):
        return _lfp(theory, mode, ref=frozenset())  # definite program: one pass
    true = frozenset()
    while True:
        over = _lfp(theory, mode, ref=true).facts       # NAF optimistic -> overestimate
        fb = _lfp(theory, mode, ref=over)               # NAF pessimistic -> underestimate
        if fb.facts == true:
            return fb
        true = frozenset(fb.facts)


def holds(fb, query, mode="owa"):
    if mode == "cwa" and is_neg(query):
        return negate(query) not in fb.facts
    return query in fb.facts


## Engine: backward chaining (SLD + negation as failure, tabling, loop check)
`reasoner/backward.py`

In [5]:
"""Backward chaining (SLD resolution with negation as failure) over Horn rules.

Goal-directed: only rules whose head unifies with the current goal are tried.
- Unification binds rule variables against goal terms and facts.
- Loop detection: a ground goal already on the ancestor stack fails (a proof
  through itself is never needed in a least model).
- Tabling: completed ground goals are memoised. A failure is memoised only if no
  loop cut happened below it, since a cut failure may succeed from another branch.
- Negation (CWA): ~p(t) succeeds iff p(t) has no proof (negation as failure).
  Rules with negated heads are not used for CWA queries (same as forward.holds).

prove(...) returns (bool, proof) where proof is a nested tuple for traces.
"""
import itertools



class BackwardChainer:
    def __init__(self, theory):
        self.facts = set(theory.facts)
        self.rules_by_pred = {}
        for r in theory.rules:
            if not is_neg(r.head):
                self.rules_by_pred.setdefault(r.head[0], []).append(r)
        self.facts_by_pred = {}
        for f in self.facts:
            self.facts_by_pred.setdefault(f[0], []).append(f)
        self.consts = {t for f in self.facts for t in f[1:]} | \
                      {t for r in theory.rules for l in (r.body + (r.head,)) for t in l[1:] if not is_var(t)}
        self.table = {}
        self._n = 0

    # -- unification of flat literals (terms are constants or variables) --------
    @staticmethod
    def _unify(pat, ground, theta):
        if pat[0] != ground[0] or len(pat) != len(ground):
            return None
        for a, b in zip(pat[1:], ground[1:]):
            a = theta.get(a, a)
            if is_var(a):
                theta = {**theta, a: b}
            elif a != b:
                return None
        return theta

    def _rename(self, rule):
        self._n += 1
        ren = {}

        def r(lit):
            return (lit[0],) + tuple(ren.setdefault(t, f"{t}_{self._n}") if is_var(t) else t for t in lit[1:])
        return tuple(r(l) for l in rule.body), r(rule.head), rule

    # -- proof search -----------------------------------------------------------
    def prove(self, goal):
        """Ground literal -> (holds, proof tree)."""
        if is_neg(goal):
            ok, _, _ = self._ground(negate(goal), ())
            return (not ok), (goal, "negation as failure", ())
        ok, proof, _ = self._ground(goal, ())
        return ok, proof

    def _ground(self, goal, stack):
        """Prove a ground positive atom. Returns (ok, proof, cut) where cut means a
        loop check pruned part of the search (so a failure must not be tabled)."""
        if goal in self.table:
            return self.table[goal] + (False,)
        if goal in self.facts:
            return True, (goal, "fact", ()), False
        if goal in stack:
            return False, None, True
        stack = stack + (goal,)
        cut_any = False
        for rule in self.rules_by_pred.get(goal[0], ()):
            body, head, orig = self._rename(rule)
            theta = self._unify(head, goal, {})
            if theta is None:
                continue
            for th, subproofs, cut in self._solve(list(body), theta, stack):
                cut_any |= cut
                proof = (goal, orig, tuple(subproofs))
                self.table[goal] = (True, proof)
                return True, proof, False
            # _solve exhausted: did any branch hit a loop cut?
            cut_any |= self._last_cut
        if not cut_any:
            self.table[goal] = (False, None)
        return False, None, cut_any

    def _solve(self, goals, theta, stack):
        """Yield (theta, proofs, cut) for a conjunction. Sets self._last_cut when exhausted."""
        self._last_cut = False
        if not goals:
            yield theta, [], False
            return
        # positive literals first so NAF literals are ground when tested
        idx = next((i for i, g in enumerate(goals) if not is_neg(g)), 0)
        lit, rest = goals[idx], goals[:idx] + goals[idx + 1:]
        lit = substitute(lit, theta)
        free = [t for t in lit[1:] if is_var(t)]
        cut_seen = False
        if is_neg(lit):
            if free:  # unsafe negation: enumerate the domain
                cands = [{**theta, free[0]: c} for c in self.consts]
                for th in cands:
                    for res in self._solve([lit] + rest, th, stack):
                        yield res
                    cut_seen |= self._last_cut
                self._last_cut = cut_seen
                return
            ok, _, cut = self._ground(negate(lit), stack)
            cut_seen |= cut
            if not ok:
                for th, ps, c in self._solve(rest, theta, stack):
                    yield th, [(lit, "negation as failure", ())] + ps, c or cut
                cut_seen |= self._last_cut
            self._last_cut = cut_seen
            return
        # positive literal: candidate ground instances from facts and rule heads
        if free:
            cands = set()
            for f in self.facts_by_pred.get(lit[0], ()):
                if self._unify(lit, f, {}) is not None:
                    cands.add(f)
            if lit[0] in self.rules_by_pred:  # heads may derive new instances: enumerate the domain
                for combo in itertools.product(sorted(self.consts), repeat=len(free)):
                    cands.add(substitute(lit, dict(zip(free, combo))))
        else:
            cands = {lit}
        for g in sorted(cands):
            th = self._unify(lit, g, theta)
            if th is None:
                continue
            ok, proof, cut = self._ground(g, stack)
            cut_seen |= cut
            if ok:
                for th2, ps, c in self._solve(rest, th, stack):
                    yield th2, [proof] + ps, c
                cut_seen |= self._last_cut
        self._last_cut = cut_seen


def proof_lines(proof, depth=0):
    goal, why, subs = proof
    label = "fact" if why == "fact" else ("negation as failure" if isinstance(why, str) else f"by {why}")
    out = ["  " * depth + f"{lit_str(goal)}   [{label}]"]
    for s in subs:
        out += proof_lines(s, depth + 1)
    return out


## Parser B + engine: structural causal models and counterfactuals
`reasoner/causal.py`

In [6]:
"""Deterministic counterfactual reasoning (CounterBench-style causal stories).

Text -> structural causal model (boolean equations over a DAG) -> Pearl's three steps:
  1. abduction:  enumerate exogenous (root) assignments consistent with observations
  2. action:     replace the equations of intervened variables (do-operator)
  3. prediction: evaluate the query variable in topological order
If several exogenous worlds remain, the answer with the larger prior mass wins
(Bern(p) priors from the text; unspecified roots default to p=0.5).
"""
import itertools
import re
from dataclasses import dataclass, field

KEYWORDS = {"not", "and", "or", "together", "cause", "causes", "caused", ","}
RE_SENT = re.compile(r"(?<=\.)\s+|(?<=\?)\s+")
RE_BERN = re.compile(r"([\w ]+?)\s*~\s*Bern\(\s*([\d.]+)\s*\)", re.I)


@dataclass
class SCM:
    eqs: dict = field(default_factory=dict)      # child -> (op, [(var, positive)], negate_output)
    priors: dict = field(default_factory=dict)   # root -> P(root = True)
    observed: dict = field(default_factory=dict)  # var -> bool

    @property
    def nodes(self):
        out = set(self.eqs) | set(self.priors) | set(self.observed)
        for _, lits, _ in self.eqs.values():
            out.update(v for v, _ in lits)
        return out

    @property
    def roots(self):
        return sorted(n for n in self.nodes if n not in self.eqs)


def _tokens(text):
    """Words and commas; consecutive non-keyword words merge into one variable name."""
    out = []
    for w in re.findall(r"[\w'-]+|,", text):
        lw = w.lower()
        if lw in KEYWORDS:
            out.append(lw)
        elif out and out[-1] not in KEYWORDS and not out[-1].startswith("\0"):
            out[-1] += "_" + lw
        else:
            out.append(lw)
    return out


def _literals(toks):
    """['not','a','and','b'] -> ('and', [('a', False), ('b', True)])"""
    op, lits, pos = "and", [], True
    for t in toks:
        if t == "not":
            pos = not pos
        elif t in ("and", "or"):
            op = t
        elif t in (",", "together"):
            continue
        else:
            lits.append((t, pos))
            pos = True
    return op, lits


def _parse_causal_sentence(sent, scm):
    toks = _tokens(sent)
    verbs = [i for i, t in enumerate(toks) if t in ("cause", "causes", "caused")]
    if not verbs:
        return False
    # LHS of each verb: scan back over literal tokens until a comma / previous verb
    starts = []
    for k in verbs:
        j = k
        while j > 0 and toks[j - 1] not in (",", "cause", "causes", "caused"):
            j -= 1
        while toks[j] in ("and", "or"):  # ", and X causes" -> drop list connective
            j += 1
        starts.append(j)
    for n, k in enumerate(verbs):
        lhs = toks[starts[n]:k]
        end = starts[n + 1] if n + 1 < len(verbs) else len(toks)
        rhs = toks[k + 1:end]
        op, causes = _literals(lhs)
        # RHS is a list of effects: "A, not B and C"
        effect, neg = [], False
        for t in rhs + [","]:
            if t == "not":
                neg = not neg
            elif t in (",", "and"):
                if effect:
                    name = "_".join(effect)
                    if name in scm.eqs:  # second mechanism for the same effect: OR them
                        o_op, o_l, o_n = scm.eqs[name]
                        scm.eqs[name] = ("or", o_l + causes, o_n)
                    else:
                        scm.eqs[name] = (op, causes, neg)
                effect, neg = [], False
            elif t not in ("or", "together"):
                effect.append(t)
    return True


def parse_story(text):
    scm = SCM()
    text = re.sub(r"(?i)\bwe know that\b", "", text)
    for m in RE_BERN.finditer(text):
        scm.priors[_tokens(m.group(1))[-1]] = float(m.group(2))
    text = RE_BERN.sub("", text)
    for sent in RE_SENT.split(text):
        s = sent.strip().rstrip(".")
        if not s:
            continue
        m = re.match(r"(?i)we observed (.+)", s)
        if m:
            obs = re.sub(r"(?i)\b(occurs?|occurred)\b", "", m.group(1))
            obs = re.sub(r"(?i)(\w+) not\b", r"not \1", obs.strip())  # "X not occurs"
            for v, pos in _literals(_tokens(obs))[1]:
                scm.observed[v] = pos
            continue
        if not _parse_causal_sentence(s, scm):
            return None
    return scm


def parse_question(q):
    """-> (target, target_positive, interventions [(var, value)], factual observations)."""
    ql = q.strip().rstrip("?").strip()
    m = re.match(r"(?i)assume (.+?), and based on this assumption, further suppose (.+?)\. would (.+?) (not )?occur$", ql)
    if m:
        ints = _literals(_tokens(m.group(1)))[1] + _literals(_tokens(m.group(2)))[1]
        return _tokens(m.group(3))[0], not m.group(4), ints, {}
    m = re.match(r"(?i)would (.+?) (not )?occur if (.+?)(?: instead of (.+))?$", ql)
    if m:
        ints = _literals(_tokens(m.group(3)))[1]
        factual = dict(_literals(_tokens(m.group(4)))[1]) if m.group(4) else {}
        return _tokens(m.group(1))[0], not m.group(2), ints, factual
    return None


def _evaluate(scm, roots_val, do, evidence=None, forced=None):
    """Evaluate all variables. ``do`` overrides equations (interventions).
    With ``evidence`` (abduction pass): an evidenced endogenous variable whose equation
    disagrees with the evidence is explained by an exogenous leak/inhibitor term; it is
    recorded in ``forced`` and keeps the evidenced value (also in the counterfactual)."""
    val = dict(roots_val)
    val.update(do)

    def get(v, stack=()):
        if v in val:
            return val[v]
        if v not in scm.eqs or v in stack:
            return False
        op, lits, neg = scm.eqs[v]
        vals = [get(p, stack + (v,)) == pos for p, pos in lits]
        out = (all(vals) if op == "and" else any(vals)) != neg
        if evidence is not None and v in evidence and evidence[v] != out:
            out = evidence[v]
            forced[v] = out
        val[v] = out
        return out

    for n in sorted(scm.nodes):
        get(n)
    return val


NOISE = 1e-3  # prior of an exogenous leak/inhibitor term being active


def counterfactual(scm, target, interventions, factual, trace=None):
    roots = scm.roots
    evidence = {**scm.observed, **factual}
    mass = {True: 0.0, False: 0.0}
    for bits in itertools.product((True, False), repeat=len(roots)):
        u = dict(zip(roots, bits))
        if any(evidence.get(r, b) != b for r, b in u.items()):
            continue
        forced = {}
        _evaluate(scm, u, {}, evidence, forced)                         # 1. abduction
        p = NOISE ** len(forced)
        for r, b in u.items():
            pr = scm.priors.get(r, 0.5)
            p *= pr if b else 1 - pr
        cf = _evaluate(scm, u, {**forced, **dict(interventions)})       # 2. action + 3. prediction
        mass[cf[target]] += p
        if trace is not None:
            trace.append((u, forced, cf[target], p))
    if mass[True] == mass[False] == 0.0:
        return None
    return mass[True] >= mass[False]


def answer(context, question, trace=None):
    scm = parse_story(context)
    q = parse_question(question)
    if scm is None or q is None:
        return None
    target, positive, ints, factual = q
    r = counterfactual(scm, target, ints, factual, trace)
    return None if r is None else (r == positive)


## First-order logic: parser, CNF, resolution refutation
`reasoner/fol.py`

In [7]:
"""First-order logic: parser (FOLIO / Prover9-like syntax), CNF conversion, and a
resolution-refutation prover.

Syntax accepted (unicode or ASCII):
  ∀x ∃y ¬ ∧ ∨ → ↔ ⊕      all x / exists y / - ~ & | -> <-> xor
  Pred(a, b)   predicates with constants / variables / functions f(x)

AST: ("atom", P, args) | ("not", f) | (op, a, b) for op in and/or/imp/iff/xor
     | ("all", v, f) | ("ex", v, f)
Terms: variables "?x", constants "c", functions ("f", t1, ...).
"""
import heapq
import itertools
import re
import time

# --------------------------------------------------------------------- parsing
_TOKEN = re.compile(r"\s*(∀|∃|¬|∧|∨|→|↔|⟷|⊕|<->|->|=>|&|\||~|\w[\w'.]*(?:-\w[\w'.]*)*|-(?!>)|!|\(|\)|,|=|≠)")
_ALIASES = {"⟷": "↔", "->": "→", "=>": "→", "<->": "↔", "&": "∧", "|": "∨", "~": "¬", "-": "¬", "!": "¬",
            "all": "∀", "forall": "∀", "exists": "∃", "xor": "⊕"}
_BIN = {"∧": ("and", 3), "∨": ("or", 2), "⊕": ("xor", 2), "→": ("imp", 1), "↔": ("iff", 0)}


class FOLSyntaxError(ValueError):
    pass


def tokenize(s):
    s = s.strip().rstrip(".")
    pos, out = 0, []
    while pos < len(s):
        m = _TOKEN.match(s, pos)
        if not m:
            if s[pos:].strip() == "":
                break
            raise FOLSyntaxError(f"bad character at {pos}: {s[pos:pos + 10]!r}")
        tok = m.group(1)
        out.append(_ALIASES.get(tok.lower(), tok) if tok.lower() in _ALIASES else tok)
        pos = m.end()
    return out


class _Parser:
    def __init__(self, toks):
        self.t, self.i, self.bound = toks, 0, []

    def peek(self):
        return self.t[self.i] if self.i < len(self.t) else None

    def eat(self, tok=None):
        cur = self.peek()
        if cur is None or (tok is not None and cur != tok):
            raise FOLSyntaxError(f"expected {tok!r}, got {cur!r}")
        self.i += 1
        return cur

    def formula(self, min_prec=0):
        left = self.unary()
        while self.peek() in _BIN and _BIN[self.peek()][1] >= min_prec:
            op, prec = _BIN[self.eat()]
            right = self.formula(prec if op == "imp" else prec + 1)  # → is right-assoc
            left = (op, left, right)
        return left

    def unary(self):
        tok = self.peek()
        if tok == "¬":
            self.eat()
            return ("not", self.unary())
        if tok in ("∀", "∃"):
            self.eat()
            v = self.eat()
            if not re.fullmatch(r"\w+", v):
                raise FOLSyntaxError(f"bad variable {v!r}")
            self.bound.append(v)
            body = self.unary()  # scope: parenthesised formula, quantifier, ¬ or atom
            self.bound.pop()
            return ("all" if tok == "∀" else "ex", "?" + v, body)
        if tok == "(":
            self.eat("(")
            f = self.formula()
            self.eat(")")
            return f
        return self.atom()

    def term(self):
        name = self.eat()
        if not re.fullmatch(r"[\w'.-]+", name):
            raise FOLSyntaxError(f"bad term {name!r}")
        if self.peek() == "(":
            self.eat("(")
            args = [self.term()]
            while self.peek() == ",":
                self.eat(",")
                args.append(self.term())
            self.eat(")")
            return (name.lower(),) + tuple(args)
        return "?" + name if name in self.bound else name.lower()

    def atom(self):
        name = self.eat()
        if not re.fullmatch(r"[\w'.-]+", name):
            raise FOLSyntaxError(f"bad predicate {name!r}")
        args = ()
        if self.peek() == "(":
            self.eat("(")
            args = [self.term()]
            while self.peek() == ",":
                self.eat(",")
                args.append(self.term())
            self.eat(")")
            args = tuple(args)
        if self.peek() in ("=", "≠"):  # equality as a plain predicate
            op = self.eat()
            other = self.term()
            lhs = args[0] if args else ("?" + name if name in self.bound else name.lower())
            a = ("atom", "=", (lhs, other))
            return ("not", a) if op == "≠" else a
        return ("atom", name, args)


FREE_VARS = ("x", "y", "z", "w", "u", "v")


def _consts(f, out):
    if f[0] == "atom":
        for a in f[2]:
            if isinstance(a, str) and not a.startswith("?"):
                out.add(a)
    elif f[0] == "not":
        _consts(f[1], out)
    elif f[0] in ("all", "ex"):
        _consts(f[2], out)
    else:
        _consts(f[1], out)
        _consts(f[2], out)
    return out


def _bind(f, name):
    """Replace constant ``name`` by variable ?name (used to close free variables)."""
    if f[0] == "atom":
        return ("atom", f[1], tuple("?" + name if a == name else a for a in f[2]))
    if f[0] == "not":
        return ("not", _bind(f[1], name))
    if f[0] in ("all", "ex"):
        return (f[0], f[1], _bind(f[2], name))
    return (f[0], _bind(f[1], name), _bind(f[2], name))


def _parse_tokens(toks):
    p = _Parser(toks)
    f = p.formula()
    if p.peek() is not None:
        raise FOLSyntaxError(f"trailing tokens: {p.t[p.i:]}")
    # variables left free by sloppy scoping (x, y, ...) are universally closed
    for v in sorted(_consts(f, set()) & set(FREE_VARS), reverse=True):
        f = ("all", "?" + v, _bind(f, v))
    return f


def parse_fol(s, repair=True):
    """Parse a formula. With ``repair``, an unbalanced formula is retried with the
    surplus ')' removed or the missing ')' appended (common annotation/LLM slip)."""
    toks = tokenize(s)
    try:
        return _parse_tokens(toks)
    except FOLSyntaxError:
        depth = toks.count("(") - toks.count(")")
        if not repair or depth == 0:
            raise
        if depth > 0:
            return _parse_tokens(toks + [")"] * depth)
        # missing "(" right after the quantifier prefix: ∀x (A ∧ B) → C)
        q = 0
        while q + 1 < len(toks) and toks[q] in ("∀", "∃"):
            q += 2
        if q:
            try:
                return _parse_tokens(toks[:q] + ["("] * -depth + toks[q:])
            except FOLSyntaxError:
                pass
        # drop the first surplus ')' that closes nothing / the trailing ones
        fixed, open_ = [], 0
        for t in toks:
            if t == ")" and open_ == 0:
                continue
            open_ += (t == "(") - (t == ")")
            fixed.append(t)
        return _parse_tokens(fixed)


# ------------------------------------------------------------------------- CNF
def _elim(f):
    """Remove →, ↔, ⊕."""
    op = f[0]
    if op == "atom":
        return f
    if op == "not":
        return ("not", _elim(f[1]))
    if op in ("all", "ex"):
        return (op, f[1], _elim(f[2]))
    a, b = _elim(f[1]), _elim(f[2])
    if op == "imp":
        return ("or", ("not", a), b)
    if op == "iff":
        return ("and", ("or", ("not", a), b), ("or", a, ("not", b)))
    if op == "xor":
        return ("and", ("or", a, b), ("or", ("not", a), ("not", b)))
    return (op, a, b)


def _nnf(f, neg=False):
    op = f[0]
    if op == "atom":
        return ("not", f) if neg else f
    if op == "not":
        return _nnf(f[1], not neg)
    if op in ("all", "ex"):
        q = op if not neg else ("ex" if op == "all" else "all")
        return (q, f[1], _nnf(f[2], neg))
    if op in ("and", "or"):
        o = op if not neg else ("or" if op == "and" else "and")
        return (o, _nnf(f[1], neg), _nnf(f[2], neg))
    raise ValueError(op)


_fresh = itertools.count()


def _subst_term(t, th):
    if isinstance(t, tuple):
        return (t[0],) + tuple(_subst_term(a, th) for a in t[1:])
    return th.get(t, t)


def _skolem(f, univ, th):
    """Standardise apart + Skolemise + drop universals (NNF input)."""
    op = f[0]
    if op == "atom":
        return ("atom", f[1], tuple(_subst_term(a, th) for a in f[2]))
    if op == "not":
        return ("not", _skolem(f[1], univ, th))
    if op == "all":
        v = f"?v{next(_fresh)}"
        return _skolem(f[2], univ + [v], {**th, f[1]: v})
    if op == "ex":
        n = next(_fresh)
        sk = (f"sk{n}",) + tuple(univ) if univ else f"sk{n}"
        return _skolem(f[2], univ, {**th, f[1]: sk})
    return (op, _skolem(f[1], univ, th), _skolem(f[2], univ, th))


def _term_vars(t):
    if isinstance(t, tuple):
        return set().union(*(_term_vars(a) for a in t[1:])) if len(t) > 1 else set()
    return {t} if t[0] == "?" else set()


def _fv(f):
    op = f[0]
    if op == "atom":
        return set().union(*(_term_vars(a) for a in f[2])) if f[2] else set()
    if op == "not":
        return _fv(f[1])
    if op in ("all", "ex"):
        return _fv(f[2]) - {f[1]}
    return _fv(f[1]) | _fv(f[2])


def _is_literal(f):
    return f[0] == "atom" or (f[0] == "not" and f[1][0] == "atom")


def _name(f, defs):
    """Tseitin-style naming: non-literal operands of ⊕/↔ are replaced by a fresh
    predicate over their free variables, with ∀vars (Def(vars) ↔ operand) added to
    ``defs``. Without this a chain A ⊕ B ⊕ C ⊕ ... grows exponentially in CNF."""
    op = f[0]
    if op == "atom":
        return f
    if op == "not":
        return ("not", _name(f[1], defs))
    if op in ("all", "ex"):
        return (op, f[1], _name(f[2], defs))
    a, b = _name(f[1], defs), _name(f[2], defs)
    if op in ("xor", "iff"):
        named = []
        for sub in (a, b):
            if _is_literal(sub):
                named.append(sub)
                continue
            vs = tuple(sorted(_fv(sub)))
            atom = ("atom", f"def{next(_fresh)}", vs)
            d = ("iff", atom, sub)
            for v in vs:
                d = ("all", v, d)
            defs.append(d)
            named.append(atom)
        a, b = named
    return (op, a, b)


DIST_LIMIT = 32  # above this many clauses, name a side instead of distributing ∨ over ∧


def _cnf(f):
    """-> list of clauses; clause = frozenset of literals (positive, pred, args)."""
    op = f[0]
    if op == "atom":
        return [frozenset([(True, f[1], f[2])])]
    if op == "not":
        a = f[1]
        return [frozenset([(False, a[1], a[2])])]
    if op == "and":
        return _cnf(f[1]) + _cnf(f[2])
    left, right = _cnf(f[1]), _cnf(f[2])
    if len(left) > 1 and len(right) > 1 and len(left) * len(right) > DIST_LIMIT:
        # (S ∨ O) with S = ∧ of clauses: introduce d with d → S, keep O ∨ d (equisatisfiable)
        side, other = (left, right) if len(left) >= len(right) else (right, left)
        vs = tuple(sorted(set().union(*(_term_vars(a) for c in side for _, _, args in c for a in args))))
        name = f"dist{next(_fresh)}"
        return [c | {(True, name, vs)} for c in other] + [c | {(False, name, vs)} for c in side]
    return [a | b for a in left for b in right]


def to_clauses(f):
    defs = []
    g = _name(f, defs)
    out = []
    for h in [g] + defs:
        out += [c for c in _cnf(_skolem(_nnf(_elim(h)), [], {})) if not _tautology(c)]
    return out


def equality_axioms(clauses):
    """Reflexivity, symmetry, transitivity and substitution axioms for '=' (only
    added when the problem uses equality)."""
    preds = {(p, len(a)) for c in clauses for _, p, a in c}
    if not any(p == "=" for p, _ in preds):
        return []
    ax = [frozenset([(True, "=", ("?e1", "?e1"))]),
          frozenset([(False, "=", ("?e1", "?e2")), (True, "=", ("?e2", "?e1"))]),
          frozenset([(False, "=", ("?e1", "?e2")), (False, "=", ("?e2", "?e3")), (True, "=", ("?e1", "?e3"))])]
    for p, n in preds:
        if p == "=":
            continue
        for i in range(n):
            xs = [f"?a{j}" for j in range(n)]
            ys = list(xs)
            ys[i] = "?b"
            ax.append(frozenset([(False, "=", (xs[i], "?b")), (False, p, tuple(xs)), (True, p, tuple(ys))]))
    return ax


def _tautology(c):
    return any((not s, p, a) in c for s, p, a in c)


# ------------------------------------------------------------------ resolution
def _is_var(t):
    return isinstance(t, str) and t[0] == "?"


def _walk(t, th):
    while _is_var(t) and t in th:
        t = th[t]
    return t


def _occurs(v, t, th):
    t = _walk(t, th)
    if t == v:
        return True
    return isinstance(t, tuple) and any(_occurs(v, a, th) for a in t[1:])


def unify(a, b, th):
    a, b = _walk(a, th), _walk(b, th)
    if a == b:
        return th
    if _is_var(a):
        return None if _occurs(a, b, th) else {**th, a: b}
    if _is_var(b):
        return None if _occurs(b, a, th) else {**th, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            th = unify(x, y, th)
            if th is None:
                return None
        return th
    return None


def _resolve_term(t, th):
    t = _walk(t, th)
    if isinstance(t, tuple):
        return (t[0],) + tuple(_resolve_term(a, th) for a in t[1:])
    return t


def _apply(clause, th):
    return frozenset((s, p, tuple(_resolve_term(a, th) for a in args)) for s, p, args in clause)


def _rename(clause):
    n = next(_fresh)
    ren = {}

    def r(t):
        if _is_var(t):
            return ren.setdefault(t, f"{t}_{n}")
        if isinstance(t, tuple):
            return (t[0],) + tuple(r(a) for a in t[1:])
        return t
    return frozenset((s, p, tuple(r(a) for a in args)) for s, p, args in clause)


def _subsumes(c, d):
    """True if clause c theta-subsumes d (c·θ ⊆ d)."""
    if len(c) > len(d):
        return False
    lits = sorted(c, key=lambda l: -len(l[2]))

    def go(i, th):
        if i == len(lits):
            return True
        s, p, args = lits[i]
        for s2, p2, args2 in d:
            if s2 == s and p2 == p and len(args2) == len(args):
                t2 = th
                for a, b in zip(args, args2):
                    # one-way matching: only c's variables may bind (no chasing into d)
                    if _is_var(a):
                        if a in t2 and t2[a] != b:
                            t2 = None
                            break
                        t2 = {**t2, a: b}
                    elif a != b:
                        t2 = None
                        break
                if t2 is not None and go(i + 1, t2):
                    return True
        return False
    return go(0, {})


def _weight(c):
    def tw(t):
        return 1 + sum(tw(a) for a in t[1:]) if isinstance(t, tuple) else 1
    return sum(1 + sum(tw(a) for a in args) for _, _, args in c)


MAX_TERM_DEPTH = 4


def _depth(t):
    return 1 + max((_depth(a) for a in t[1:]), default=0) if isinstance(t, tuple) else 0


def _canon(c):
    """Rename variables in order of appearance after a variable-blind literal sort,
    so clauses that are variants of each other compare equal."""
    def shape(t):
        if isinstance(t, tuple):
            return (t[0],) + tuple(shape(a) for a in t[1:])
        return "?" if _is_var(t) else t
    lits = sorted(c, key=lambda l: (l[0], l[1], repr(tuple(shape(a) for a in l[2]))))
    ren = {}

    def r(t):
        if isinstance(t, tuple):
            return (t[0],) + tuple(r(a) for a in t[1:])
        if _is_var(t):
            return ren.setdefault(t, f"?c{len(ren)}")
        return t
    return frozenset((sg, p, tuple(r(a) for a in args)) for sg, p, args in lits)


def _unify_args(args, args2, th=None):
    th = {} if th is None else th
    for a, b in zip(args, args2):
        th = unify(a, b, th)
        if th is None:
            return None
    return th


def _factors(c):
    """Binary factors: unify two same-sign literals of the same predicate."""
    lits = list(c)
    for i in range(len(lits)):
        for j in range(i + 1, len(lits)):
            (s1, p1, a1), (s2, p2, a2) = lits[i], lits[j]
            if s1 == s2 and p1 == p2 and len(a1) == len(a2):
                th = _unify_args(a1, a2)
                if th is not None:
                    yield _apply(c - {lits[j]}, th)


def refute(axioms, sos, max_clauses=6000, timeout=2.0, trace=None):
    """Given-clause resolution with set-of-support, best-first (lightest clause)
    selection, binary factoring, variant/tautology/subsumption deletion and a
    term-depth bound. Returns True if the empty clause is derived (unsatisfiable),
    False if saturated (satisfiable), None on resource limit."""
    deadline = time.perf_counter() + timeout
    usable, seen, heap, parents = [], set(), [], {}
    limited = False
    for c in axioms:
        c = _canon(c)
        if c not in seen:
            seen.add(c)
            usable.append(c)
    for c in sos:
        if not c:
            return True
        c = _canon(c)
        if c not in seen:
            seen.add(c)
            heapq.heappush(heap, (_weight(c), next(_fresh), c))
    units = [u for u in usable if len(u) == 1]

    def add(res, given, other):
        nonlocal limited
        if _tautology(res):
            return False
        if any(_depth(a) > MAX_TERM_DEPTH for _, _, args in res for a in args):
            limited = True
            return False
        res = _canon(res)
        if res in seen or any(_subsumes(u, res) for u in units):
            return False
        seen.add(res)
        if trace is not None:
            parents[res] = (given, other)
        if not res:
            if trace is not None:
                trace.extend(_proof(res, parents))
            return True
        heapq.heappush(heap, (_weight(res), next(_fresh), res))
        return False

    while heap:
        if len(seen) > max_clauses or time.perf_counter() > deadline:
            return None
        _, _, given = heapq.heappop(heap)
        if any(_subsumes(u, given) for u in usable if len(u) <= len(given)):
            continue
        for f in _factors(given):
            if add(f, given, given):
                return True
        g = _rename(given)
        for other in usable + [given]:
            o = _rename(other)
            for lit in g:
                s, p, args = lit
                for lit2 in o:
                    s2, p2, args2 = lit2
                    if s2 == s or p2 != p or len(args2) != len(args):
                        continue
                    th = _unify_args(args, args2)
                    if th is None:
                        continue
                    if add(_apply((g - {lit}) | (o - {lit2}), th), given, other):
                        return True
        usable.append(given)
        if len(given) == 1:
            units.append(given)
    return None if limited else False


def _proof(empty, parents):
    out, stack, done = [], [empty], set()
    while stack:
        c = stack.pop()
        if c in done or c not in parents:
            continue
        done.add(c)
        a, b = parents[c]
        out.append((a, b, c))
        stack.extend([a, b])
    return list(reversed(out))


def clause_str(c):
    if not c:
        return "□"

    def t(x):
        if isinstance(x, tuple):
            return f"{x[0]}({', '.join(map(t, x[1:]))})"
        return x.lstrip("?")
    return " ∨ ".join(("" if s else "¬") + f"{p}({', '.join(map(t, a))})" for s, p, a in sorted(c, key=str))


def entails(premises, conclusion, **kw):
    """premises: list of FOL ASTs; conclusion: AST. True iff premises ⊨ conclusion
    (refutation found); False otherwise (including resource limit)."""
    axioms = [c for f in premises for c in to_clauses(f)]
    negated = to_clauses(("not", conclusion))
    axioms += equality_axioms(axioms + negated)
    return refute(axioms, negated, **kw) is True


def three_way(premises, conclusion, **kw):
    """'True' if KB ⊨ φ, 'False' if KB ⊨ ¬φ, else 'Uncertain'."""
    if entails(premises, conclusion, **kw):
        return "True"
    if entails(premises, ("not", conclusion), **kw):
        return "False"
    return "Uncertain"


## Parser C: LLM semantic parser (translation only, never answers)
`reasoner/translator.py`

In [8]:
"""LLM-based semantic parser: natural language -> first-order logic.

The LLM is used ONLY to translate sentences into FOL formulas. It is never asked
for, and its output is never read as, a True/False answer: the decision is made
by our own resolution prover (reasoner.fol) on the parsed formulas.

Pipeline per problem: prompt -> k samples -> parse/validate with our FOL parser
-> one repair round for invalid samples (parser error fed back) -> cache.
"""
import hashlib
import json
import os
import re


SYSTEM = """You are a semantic parser. Translate each natural-language premise and the conclusion into first-order logic.
Rules:
- Use ∀ ∃ ¬ ∧ ∨ → ↔ ⊕ and parentheses. Put the quantified formula in parentheses: ∀x (P(x) → Q(x)).
- Predicates are CamelCase (Dog, LivesIn); constants are lowercase_with_underscores (rex, new_york); variables are x, y, z.
- Use the SAME predicate name for the same concept everywhere. Give every predicate a fixed arity.
- "either A or B" (exclusive) is ⊕; "A or B" is ∨; "if and only if" is ↔.
- Translate the conclusion faithfully; do not decide whether it follows.
Output exactly this format and nothing else:
PREMISES:
<one formula per line, same order as the premises>
CONCLUSION:
<one formula>"""

# Hand-written examples (not taken from any evaluation set).
FEWSHOT = [
    (["All dogs are animals.", "Rex is a dog.", "Every animal that is hungry eats something."],
     "Rex eats something if he is hungry.",
     ["∀x (Dog(x) → Animal(x))", "Dog(rex)", "∀x ((Animal(x) ∧ Hungry(x)) → ∃y Eats(x, y))"],
     "Hungry(rex) → ∃y Eats(rex, y)"),
    (["A vehicle is either electric or uses fuel.", "No electric vehicle emits smoke.",
      "The Volt is a vehicle that does not use fuel.", "Some cars made in Detroit are vehicles."],
     "The Volt emits smoke.",
     ["∀x (Vehicle(x) → (Electric(x) ⊕ UsesFuel(x)))", "∀x (Electric(x) → ¬EmitsSmoke(x))",
      "Vehicle(volt) ∧ ¬UsesFuel(volt)", "∃x (Car(x) ∧ MadeIn(x, detroit) ∧ Vehicle(x))"],
     "EmitsSmoke(volt)"),
    (["Anyone who teaches a course at a university is an employee of that university.",
      "Maria teaches Logic 101 at Riverside University.", "Logic 101 is a course.",
      "A person is a student if and only if they are enrolled in some program."],
     "Maria is an employee of Riverside University and Maria is not a student.",
     ["∀x ∀y ∀z ((Teaches(x, y, z) ∧ Course(y) ∧ University(z)) → EmployeeOf(x, z))",
      "Teaches(maria, logic_101, riverside_university) ∧ University(riverside_university)", "Course(logic_101)",
      "∀x (Student(x) ↔ ∃y EnrolledIn(x, y))"],
     "EmployeeOf(maria, riverside_university) ∧ ¬Student(maria)"),
]


def _user_msg(premises, conclusion):
    lines = "\n".join(f"{i + 1}. {p}" for i, p in enumerate(premises))
    return f"Premises:\n{lines}\nConclusion:\n{conclusion}"


def _asst_msg(fol_premises, fol_conclusion):
    return "PREMISES:\n" + "\n".join(fol_premises) + "\nCONCLUSION:\n" + fol_conclusion


def build_messages(premises, conclusion, error=None, previous=None, fewshot=True):
    msgs = [{"role": "system", "content": SYSTEM}]
    for p, c, fp, fc in (FEWSHOT if fewshot else []):
        msgs += [{"role": "user", "content": _user_msg(p, c)},
                 {"role": "assistant", "content": _asst_msg(fp, fc)}]
    msgs.append({"role": "user", "content": _user_msg(premises, conclusion)})
    if error is not None:  # repair round: show the parser error, ask for a corrected translation
        msgs += [{"role": "assistant", "content": previous},
                 {"role": "user", "content": f"Our FOL parser rejected that output: {error}\n"
                                             "Output the corrected translation in the same format."}]
    return msgs


def split_premises(context):
    """Context string or list -> list of premise sentences."""
    if isinstance(context, list):
        return [p.strip() for p in context if p.strip()]
    parts = [p.strip() for p in context.split("\n") if p.strip()]
    if len(parts) <= 1:
        parts = [p.strip() for p in re.split(r"(?<=[.!?])\s+", context) if p.strip()]
    return parts


def _strip_num(line):
    return re.sub(r"^\s*(?:\d+[.)]|[-*•])\s*", "", line).strip()


def _arity_check(formulas):
    arity = {}

    def walk(f):
        if f[0] == "atom":
            if arity.setdefault(f[1], len(f[2])) != len(f[2]):
                raise FOLSyntaxError(f"predicate {f[1]} used with arities {arity[f[1]]} and {len(f[2])}")
        elif f[0] == "not":
            walk(f[1])
        elif f[0] in ("all", "ex"):
            walk(f[2])
        else:
            walk(f[1])
            walk(f[2])
    for f in formulas:
        walk(f)


def parse_output(text, n_premises=None):
    """LLM text -> (premise ASTs, conclusion AST). Raises FOLSyntaxError."""
    m = re.search(r"PREMISES:\s*(.*?)\s*CONCLUSION:\s*(.+)", text, re.S)
    if not m:
        raise FOLSyntaxError("missing PREMISES:/CONCLUSION: sections")
    prem_lines = [_strip_num(l) for l in m.group(1).splitlines() if _strip_num(l)]
    concl_line = _strip_num(m.group(2).strip().splitlines()[0])
    prem = []
    for i, line in enumerate(prem_lines):
        try:
            prem.append(parse_fol(line))
        except FOLSyntaxError as e:
            raise FOLSyntaxError(f"premise {i + 1} '{line}': {e}")
    try:
        concl = parse_fol(concl_line)
    except FOLSyntaxError as e:
        raise FOLSyntaxError(f"conclusion '{concl_line}': {e}")
    _arity_check(prem + [concl])
    return prem, concl


class TranslationCache:
    def __init__(self, path):
        self.path = path
        self.data = json.load(open(path)) if path and os.path.exists(path) else {}

    @staticmethod
    def key(model, premises, conclusion, k):
        h = hashlib.sha1(json.dumps([model, SYSTEM, premises, conclusion, k]).encode()).hexdigest()
        return h

    def save(self):
        if self.path:
            os.makedirs(os.path.dirname(self.path) or ".", exist_ok=True)
            json.dump(self.data, open(self.path, "w"))


class VLLMBackend:
    """Batched local generation. Sample 0 is greedy; samples 1..k-1 use temperature."""

    def __init__(self, model, adapter=None, tp=1, max_model_len=8192, gpu_mem=0.88):
        from vllm import LLM
        self.name = model + (f"+{adapter}" if adapter else "")
        self.adapter = adapter
        self.fewshot = adapter is None  # fine-tuned adapters are trained zero-shot
        lora_kw = dict(max_num_seqs=64, enforce_eager=True) if adapter else {}
        if adapter:  # zero-shot prompts are short; LoRA buffers need headroom
            max_model_len, gpu_mem = min(max_model_len, 4096), min(gpu_mem, 0.85)
        self.llm = LLM(model=model, tensor_parallel_size=tp, max_model_len=max_model_len, **lora_kw,
                       gpu_memory_utilization=gpu_mem, enable_lora=adapter is not None, swap_space=16,
                       max_lora_rank=64 if adapter else None)
        self.tok = self.llm.get_tokenizer()

    def generate(self, message_lists, k=1, temperature=0.7, max_tokens=1024):
        from vllm import SamplingParams
        prompts = [self.tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in message_lists]
        lora = None
        if self.adapter:
            from vllm.lora.request import LoRARequest
            lora = LoRARequest("adapter", 1, self.adapter)
        greedy = self.llm.generate(prompts, SamplingParams(temperature=0, max_tokens=max_tokens),
                                   lora_request=lora, use_tqdm=False)
        outs = [[o.outputs[0].text] for o in greedy]
        if k > 1:
            sampled = self.llm.generate(prompts, SamplingParams(n=k - 1, temperature=temperature, top_p=0.95,
                                                                max_tokens=max_tokens, seed=0),
                                        lora_request=lora, use_tqdm=False)
            for o, s in zip(outs, sampled):
                o.extend(c.text for c in s.outputs)
        return outs


def translate(problems, backend, k=1, cache=None, repair=True, return_text=False, model_name=None):
    """problems: list of (premise sentences, conclusion sentence).
    Returns per problem a list of k candidates: (premise ASTs, conclusion AST) or None.
    Without a backend, ``model_name`` (same format as VLLMBackend.name) selects cached
    translations, so a CPU-only run can reuse a shipped cache."""
    results = [None] * len(problems)
    todo = []
    for i, (prem, concl) in enumerate(problems):
        key = TranslationCache.key(backend.name if backend else (model_name or ""), prem, concl, k)
        if cache is not None and key in cache.data:
            results[i] = cache.data[key]
        else:
            todo.append((i, key))
    if todo and backend is not None:
        fs = getattr(backend, "fewshot", True)
        outs = backend.generate([build_messages(*problems[i], fewshot=fs) for i, _ in todo], k=k)
        for (i, key), texts in zip(todo, outs):
            results[i] = list(texts)
        if repair:  # one repair round for invalid samples, batched
            fix = []
            for i, _ in todo:
                for j, text in enumerate(results[i]):
                    try:
                        parse_output(text)
                    except FOLSyntaxError as e:
                        fix.append((i, j, build_messages(*problems[i], error=str(e), previous=text, fewshot=fs)))
            if fix:
                fixed = backend.generate([m for _, _, m in fix], k=1)
                for (i, j, _), texts in zip(fix, fixed):
                    results[i][j] = texts[0]
        if cache is not None:
            for i, key in todo:
                cache.data[key] = results[i]
            cache.save()
    parsed = []
    for texts in results:
        cands = []
        for text in texts or []:
            try:
                cands.append(parse_output(text))
            except FOLSyntaxError:
                cands.append(None)
        parsed.append(cands)
    return (parsed, results) if return_text else parsed


## Router: end-to-end pipeline
`reasoner/pipeline.py`

In [9]:
"""End-to-end router: (context, statement) -> True/False via classical reasoning.

Route per item, cheapest first:
  1. causal   : causal story + counterfactual question  -> SCM (abduction/action/prediction)
  2. horn     : every sentence parsed by the grammar     -> backward chaining, CWA (NAF)
  3. hybrid   : most sentences parsed; LLM translates the rest into FOL, converted to
                Horn rules with the grammar's naming    -> backward chaining, CWA
     (backward chaining with tabling agrees with forward chaining on all public RuleTaker
      questions and is ~3x faster per query; forward chaining is used for mode="owa")
  4. fol      : LLM translates everything into FOL (k samples) -> resolution prover,
                majority vote over the prover's 3-way verdicts (OWA)
  5. default  : nothing worked -> False (not entailed)
"""
import os
import re
import time
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor


HYBRID_MIN_PARSED = 0.6  # fraction of sentences the grammar must cover to stay in Horn/CWA mode
# Generic nouns the grammar reads as the unrestricted domain ("someone", "things", "people").
# An LLM may reify them as predicates (Person(x) ∧ Round(x) → ...); such a body literal is
# dropped when the theory never states the predicate, so both parsers share one convention.
SORTALS = {"person", "people", "thing", "things", "someone", "something", "somebody",
           "entity", "individual", "object", "one", "being"}


# ------------------------------------------------------------ FOL -> Horn (hybrid)
def _norm_pred(name, arity):
    p = re.sub(r"(?<!^)(?=[A-Z])", "_", name).lower()  # CamelCase -> snake
    p = re.sub(r"^(is|are)_", "", p)
    return lemma(p) if arity == 2 else p


def _norm_const(c):
    c = re.sub(r"(?<!^)(?=[A-Z])", "_", c).lower()  # baldEagle -> bald_eagle (grammar naming)
    return c[4:] if c.startswith("the_") else c


def _lit(f, var_map):
    neg = False
    while f[0] == "not":
        neg, f = not neg, f[1]
    if f[0] != "atom" or not f[2] or any(isinstance(a, tuple) for a in f[2]):
        return None
    args = tuple("?x" if a in var_map else _norm_const(a) for a in f[2])
    p = _norm_pred(f[1], len(args))
    return ("~" + p if neg else p,) + args


def _conj(f, var_map):
    if f[0] == "and":
        a, b = _conj(f[1], var_map), _conj(f[2], var_map)
        return a + b if a is not None and b is not None else None
    lit = _lit(f, var_map)
    return [lit] if lit else None


def fol_to_horn(f):
    """FOL AST of a fact or a definite rule -> ('fact', lits) | ('rule', [Rule]) | None."""
    var_map = set()
    while f[0] == "all":
        var_map.add(f[1])
        f = f[2]
    if len(var_map) > 1:
        return None
    if f[0] == "imp":
        body, head = _conj(f[1], var_map), _conj(f[2], var_map)
        if body and head:
            return "rule", [Rule(tuple(body), h) for h in head]
        return None
    lits = _conj(f, var_map)
    if lits and not var_map:
        return "fact", lits
    return None


def vote_entailed(cands):
    """Majority vote over k translations: does each sample's KB entail its conclusion?
    The task's label is binary, so one refutation (KB |= phi) per sample suffices; proving
    KB |= ~phi would only separate False from Uncertain. Ties count as "not entailed"; for
    k=3 this equals the plurality of 3-way verdicts with ties towards "not entailed".
    Returns None when no sample produced valid formulas."""
    votes = [entails(c[0], c[1], timeout=1.0) for c in cands if c is not None]
    return sum(votes) * 2 > len(votes) if votes else None


# ------------------------------------------------------------------ the router
class Reasoner:
    def __init__(self, backend=None, cache=None, k=5, mode="cwa", model_name=None):
        self.backend, self.cache, self.k, self.mode = backend, cache, k, mode
        self.model_name = model_name  # cache key when running without a backend
        self.stats = Counter()
        self.timing = defaultdict(float)

    def solve(self, items):
        """items: list of dicts with id, context, statement -> {id: bool}, {id: route}."""
        preds, routes = {}, {}
        by_ctx = defaultdict(list)
        for it in items:
            by_ctx[it["context"]].append(it)

        need_llm = []  # (kind, context, items, theory)
        for ctx, its in by_ctx.items():
            t0 = time.perf_counter()
            # 1. causal
            if re.search(r"\bcauses?\b", ctx) and all(parse_question(i["statement"]) for i in its):
                for it in its:
                    r = answer(ctx, it["statement"])
                    if r is not None:
                        preds[it["id"]], routes[it["id"]] = r, "causal"
                its = [i for i in its if i["id"] not in preds]
                if not its:
                    self.timing["reason"] += time.perf_counter() - t0
                    continue
            # 2./3. grammar
            th = parse_context(ctx)
            n_sent = len([s for s in RE_SENT.split(ctx.strip()) if s.strip()])
            queries = {i["id"]: parse_query(i["statement"]) for i in its}
            if not th.unparsed and all(q is not None for q in queries.values()):
                ask = self._engine(th)
                for it in its:
                    preds[it["id"]], routes[it["id"]] = ask(queries[it["id"]]), "horn"
            elif n_sent and 1 - len(th.unparsed) / n_sent >= HYBRID_MIN_PARSED \
                    and all(q is not None for q in queries.values()):
                need_llm.append(("hybrid", ctx, its, th))
            else:
                need_llm.append(("fol", ctx, its, None))
            self.timing["reason"] += time.perf_counter() - t0

        if need_llm:
            self._llm_routes(need_llm, preds, routes)
        for it in items:
            if it["id"] not in preds:
                preds[it["id"]], routes[it["id"]] = False, "default"
        self.stats.update(routes.values())
        return preds, routes

    def _engine(self, theory):
        """Query function for a Horn theory: tabled backward chaining under CWA,
        forward chaining (saturate once, then look up) under OWA."""
        if self.mode == "cwa":
            bc = BackwardChainer(theory)
            return lambda q: bc.prove(q)[0]
        fb = forward_chain(theory, self.mode)
        return lambda q: holds(fb, q, self.mode)

    def _llm_routes(self, jobs, preds, routes):
        problems, index = [], []
        for kind, ctx, its, th in jobs:
            if kind == "hybrid":
                # translate only the sentences the grammar could not parse
                problems.append((list(th.unparsed), its[0]["statement"]))
                index.append((kind, ctx, its, th, None))
            else:
                prem = split_premises(ctx)
                for it in its:
                    problems.append((prem, it["statement"]))
                    index.append((kind, ctx, [it], th, None))
        t0 = time.perf_counter()
        cands = translate(problems, self.backend, k=self.k, cache=self.cache, model_name=self.model_name) \
            if (self.backend or self.cache) else [[] for _ in problems]
        self.timing["llm"] += time.perf_counter() - t0

        t0 = time.perf_counter()
        fol = [(its[0], cs) for (kind, _, its, _, _), cs in zip(index, cands) if kind == "fol"]
        if len(fol) >= 16:  # independent refutations: run them in parallel
            with ProcessPoolExecutor(max_workers=min(16, os.cpu_count() or 1)) as ex:
                verdicts = list(ex.map(vote_entailed, [cs for _, cs in fol], chunksize=2))
        else:
            verdicts = [vote_entailed(cs) for _, cs in fol]
        for (it, _), v in zip(fol, verdicts):
            if v is not None:
                preds[it["id"]], routes[it["id"]] = v, "fol"
        for (kind, ctx, its, th, _), cs in zip(index, cands):
            if kind == "hybrid":
                self._hybrid(its, th, cs, preds, routes)
        self.timing["reason"] += time.perf_counter() - t0

    def _hybrid(self, its, th, cands, preds, routes):
        # use the first candidate whose formulas all convert to Horn clauses
        for cand in cands:
            if cand is None:
                continue
            extra = Theory(set(th.facts), list(th.rules))
            ok = True
            for f in cand[0]:
                r = fol_to_horn(f)
                if r is None:
                    ok = False
                    break
                (extra.facts.update if r[0] == "fact" else extra.rules.extend)(r[1])
            if ok:
                stated = {f[0] for f in extra.facts} | {r.head[0] for r in extra.rules}
                extra.rules = [Rule(tuple(l for l in r.body if not (len(l) == 2 and l[0] in SORTALS
                                                                    and l[0] not in stated)) or r.body,
                                    r.head, r.text) for r in extra.rules]
                ask = self._engine(extra)
                for it in its:
                    preds[it["id"]] = ask(parse_query(it["statement"]))
                    routes[it["id"]] = "hybrid"
                return


## Evaluation on the public datasets
Accuracy and inference time per source.

In [10]:
def load_items(path, limit=None):
    items = []
    for i, line in enumerate(open(path)):
        if limit and i >= limit: break
        r = json.loads(line)
        for q in r['questions']:
            items.append({'id': q['qid'], 'context': r['context'], 'statement': q['statement'],
                          'label': q['label'], 'source': r['source'], 'config': r['config']})
    return items

backend = VLLMBackend(LLM_MODEL, adapter=LLM_ADAPTER, tp=LLM_TP) if USE_LLM else None
cache = TranslationCache(CACHE_FILE)
MODEL_NAME = LLM_MODEL + (f'+{LLM_ADAPTER}' if LLM_ADAPTER else '')  # cache key (CPU-only runs reuse the shipped cache)
reasoner = Reasoner(backend=backend, cache=cache, k=K_SAMPLES, model_name=MODEL_NAME)

results = {}
for name in ['ruletaker_dev', 'ruletaker_test', 'counterbench', 'folio_validation', 'ruletaker_dev_paraphrased']:
    path = os.path.join(DATA_DIR, name + '.jsonl')
    if not os.path.exists(path):
        print(f'{name}: not available'); continue
    items = load_items(path)
    t0 = time.perf_counter()
    preds, routes = reasoner.solve(items)
    dt = time.perf_counter() - t0
    acc = sum(preds[i['id']] == i['label'] for i in items) / len(items) * 100
    results[name] = (acc, dt, len(items))
    print(f'{name:18s} accuracy {acc:7.3f}%  n={len(items):6d}  time {dt:7.2f}s  routes {dict(Counter(routes.values()))}')
print('time split (s):', {k: round(v, 2) for k, v in reasoner.timing.items()},
      '' if USE_LLM else '(USE_LLM=False: LLM translations come from the shipped cache)')
n_all = sum(r[2] for r in results.values())
print(f"\nOVERALL accuracy {sum(r[0]*r[2] for r in results.values())/n_all:.3f}%  "
      f"total inference time {sum(r[1] for r in results.values()):.2f}s for {n_all} queries")

ruletaker_dev      accuracy 100.000%  n= 75872  time    1.97s  routes {'horn': 75872}


ruletaker_test     accuracy 100.000%  n=151911  time    4.02s  routes {'horn': 151911}


counterbench       accuracy  94.583%  n=  1200  time    0.25s  routes {'causal': 1200}


folio_validation   accuracy  87.685%  n=   203  time    5.98s  routes {'fol': 203}


ruletaker_dev_paraphrased accuracy  99.071%  n=   646  time    0.17s  routes {'fol': 23, 'hybrid': 623}
time split (s): {'reason': 12.12, 'llm': 0.13} (USE_LLM=False: LLM translations come from the shipped cache)

OVERALL accuracy 99.958%  total inference time 12.40s for 229832 queries


## Reasoning traces

In [11]:
def explain_horn(context, statement, mode='cwa'):
    th = parse_context(context); fb = forward_chain(th, mode); q = parse_query(statement)
    def show(f, depth=0):
        why = fb.why.get(f)
        print('  ' * depth + lit_str(f) + ('   [fact]' if why is None else f'   [by {why[0]}]'))
        if why:
            for p in why[1]:
                show(p, depth + 1) if p in fb.facts else print('  ' * (depth + 1) + lit_str(p) + '   [negation as failure]')
    print('Query:', statement, '->', holds(fb, q, mode))
    if q in fb.facts: show(q)

ctx = ('Bob is young. Bob is kind. If someone is young, then they are smart. '
       'If someone is smart and kind, then they are successful.')
print('Forward chaining (derivation record):')
explain_horn(ctx, 'Bob is successful.')
print('\nBackward chaining (goal-directed proof tree):')
ok, proof = BackwardChainer(parse_context(ctx)).prove(parse_query('Bob is successful.'))
print('\n'.join(proof_lines(proof)))
ctx2 = 'Dave is red. If something is red and not furry then it is white.'
ok, proof = BackwardChainer(parse_context(ctx2)).prove(parse_query('Dave is white.'))
print('\nNegation as failure:', ok); print('\n'.join(proof_lines(proof)))
print()
tr = []
ctx = 'We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.'
print('Would Lumbo occur if not Ziklo instead of Ziklo? ->', answer(ctx, 'Would Lumbo occur if not Ziklo instead of Ziklo?', trace=tr))
for u, forced, val, p in tr: print('  exogenous', u, 'leaks', forced, '-> counterfactual Lumbo =', val)
print()
kb = [parse_fol(x) for x in ['∀x (Dog(x) → Animal(x))', 'Dog(rex)']]
ax = [c for f in kb for c in to_clauses(f)]; tr = []
print('Resolution proof of Animal(rex):', refute(ax, to_clauses(parse_fol('¬Animal(rex)')), trace=tr))
for a, b, c in tr: print('  ', clause_str(a), ' + ', clause_str(b), ' ⟹ ', clause_str(c))

Forward chaining (derivation record):
Query: Bob is successful. -> True
successful(bob)   [by smart(?x) ∧ kind(?x) → successful(?x)]
  smart(bob)   [by young(?x) → smart(?x)]
    young(bob)   [fact]
  kind(bob)   [fact]

Backward chaining (goal-directed proof tree):
successful(bob)   [by smart(?x) ∧ kind(?x) → successful(?x)]
  smart(bob)   [by young(?x) → smart(?x)]
    young(bob)   [fact]
  kind(bob)   [fact]

Negation as failure: True
white(dave)   [by red(?x) ∧ ¬furry(?x) → white(?x)]
  red(dave)   [fact]
  ¬furry(dave)   [negation as failure]

Would Lumbo occur if not Ziklo instead of Ziklo? -> False
  exogenous {'ziklo': True} leaks {} -> counterfactual Lumbo = False

Resolution proof of Animal(rex): True
   ¬Animal(rex)  +  ¬Dog(c0) ∨ Animal(c0)  ⟹  ¬Dog(rex)
   ¬Dog(rex)  +  Dog(rex)  ⟹  □


## Generate submission.csv

In [12]:
def read_test(path):
    rows = [json.loads(l) for l in open(path)] if path.endswith('.jsonl') else list(csv.DictReader(open(path)))
    alias = {'question': 'statement', 'query': 'statement', 'hypothesis': 'statement', 'premises': 'context', 'theory': 'context'}
    out = []
    for r in rows:
        r = {alias.get(k, k): v for k, v in r.items()}
        out.append({'id': r['id'], 'context': r['context'], 'statement': r['statement']})
    return out

if TEST_FILE:
    test = read_test(TEST_FILE)
    t0 = time.perf_counter()
    preds, routes = reasoner.solve(test)
    print(f'{len(test)} queries in {time.perf_counter() - t0:.2f}s; routes {dict(Counter(routes.values()))}')
    with open('submission.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'label'])
        for it in test: w.writerow([it['id'], preds[it['id']]])
    print('wrote submission.csv')
else:
    print('Set TEST_FILE to produce submission.csv')

Set TEST_FILE to produce submission.csv
